# Search best matching TCR structures

Search RCSB PDB for the best matching polymer entities for the selected alpha and beta TCR sequences.

In [ ]:
from pathlib import Path

import pandas as pd
import requests

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "TCRTools").exists() and not (NOTEBOOK_DIR / "data").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "TCRTools"

OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

RCSB_SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
RCSB_DATA_URL = "https://data.rcsb.org/rest/v1/core"

In [ ]:
SELECTED_UNITS_TABLE = OUTPUT_DIR / "selected_tcr_units.csv"
ROWS_PER_SEQUENCE = 5
IDENTITY_CUTOFF = 0.70
EVALUE_CUTOFF = 1

selected_units = pd.read_csv(SELECTED_UNITS_TABLE)
selected_units[["tcr_unit_name", "chain_type", "abundance", "sequence_id"]]

In [ ]:
def clean_sequence(sequence):
    return "".join(str(sequence).upper().replace("-", "").split())


def first_match_context(hit):
    for service in hit.get("services", []):
        for node in service.get("nodes", []):
            contexts = node.get("match_context") or []
            if contexts:
                context = contexts[0].copy()
                context["bitscore"] = context.get("bitscore", node.get("original_score"))
                return context
    return {}


def rcsb_sequence_search(sequence, rows=10, identity_cutoff=0.7, evalue_cutoff=1):
    payload = {
        "query": {
            "type": "terminal",
            "service": "sequence",
            "parameters": {
                "evalue_cutoff": evalue_cutoff,
                "identity_cutoff": identity_cutoff,
                "target": "pdb_protein_sequence",
                "value": clean_sequence(sequence),
            },
        },
        "return_type": "polymer_entity",
        "request_options": {
            "paginate": {"start": 0, "rows": rows},
            "results_verbosity": "verbose",
            "scoring_strategy": "sequence",
        },
    }
    response = requests.post(RCSB_SEARCH_URL, json=payload, timeout=30)
    if response.status_code == 204:
        return []
    response.raise_for_status()

    hits = []
    for rank, hit in enumerate(response.json().get("result_set", []), start=1):
        entry_id, entity_id = hit["identifier"].split("_", 1)
        hits.append(
            {
                "rank": rank,
                "identifier": hit["identifier"],
                "entry_id": entry_id,
                "entity_id": entity_id,
                "score": hit.get("score"),
                **first_match_context(hit),
            }
        )
    return hits


def polymer_entity_metadata(entry_id, entity_id):
    response = requests.get(f"{RCSB_DATA_URL}/polymer_entity/{entry_id}/{entity_id}", timeout=30)
    response.raise_for_status()
    data = response.json()
    identifiers = data.get("rcsb_polymer_entity_container_identifiers", {})
    entity = data.get("rcsb_polymer_entity", {})
    return {
        "description": entity.get("pdbx_description"),
        "asym_ids": ",".join(identifiers.get("asym_ids", [])),
        "auth_asym_ids": ",".join(identifiers.get("auth_asym_ids", [])),
    }


def entry_metadata(entry_id):
    response = requests.get(f"{RCSB_DATA_URL}/entry/{entry_id}", timeout=30)
    response.raise_for_status()
    data = response.json()
    methods = [item.get("method") for item in data.get("exptl", []) if item.get("method")]
    resolutions = [item.get("ls_d_res_high") for item in data.get("refine", []) if item.get("ls_d_res_high") is not None]
    resolutions += [item.get("resolution") for item in data.get("em_3d_reconstruction", []) if item.get("resolution") is not None]
    return {
        "experimental_method": "; ".join(methods),
        "resolution": min(resolutions) if resolutions else None,
    }


def search_best_structures_for_units(units, rows=5, identity_cutoff=0.7, evalue_cutoff=1):
    records = []
    for _, unit in units.iterrows():
        for hit in rcsb_sequence_search(unit["sequence"], rows=rows, identity_cutoff=identity_cutoff, evalue_cutoff=evalue_cutoff):
            records.append(
                {
                    "clonotype_id": unit["clonotype_id"],
                    "tcr_unit_name": unit["tcr_unit_name"],
                    "chain_type": unit["chain_type"],
                    "sequence_id": unit["sequence_id"],
                    "abundance": unit["abundance"],
                    "highlight_sequence": unit.get("highlight_sequence", ""),
                    "highlight_label": unit.get("highlight_label", ""),
                    **hit,
                    **polymer_entity_metadata(hit["entry_id"], hit["entity_id"]),
                    **entry_metadata(hit["entry_id"]),
                }
            )
    if not records:
        return pd.DataFrame()
    return pd.DataFrame(records).sort_values(["chain_type", "sequence_id", "rank", "resolution"], na_position="last")


def best_hit_per_unit(search_results):
    return (
        search_results.sort_values(["chain_type", "sequence_id", "rank", "resolution"], na_position="last")
        .groupby(["clonotype_id", "sequence_id"], as_index=False, group_keys=False)
        .head(1)
        .reset_index(drop=True)
    )

In [ ]:
search_results = search_best_structures_for_units(
    selected_units,
    rows=ROWS_PER_SEQUENCE,
    identity_cutoff=IDENTITY_CUTOFF,
    evalue_cutoff=EVALUE_CUTOFF,
)
if search_results.empty:
    raise ValueError("No RCSB/PDB sequence-search results found")

search_results.to_csv(OUTPUT_DIR / "rcsb_search_results.csv", index=False)
columns = [
    "tcr_unit_name", "chain_type", "rank", "identifier", "description", "auth_asym_ids",
    "sequence_identity", "evalue", "bitscore", "experimental_method", "resolution",
]
search_results[columns]

In [ ]:
best_hits = best_hit_per_unit(search_results)
best_hits.to_csv(OUTPUT_DIR / "best_tcr_structure_hits.csv", index=False)
best_hits[["tcr_unit_name", "chain_type", "identifier", "auth_asym_ids", "sequence_identity", "resolution"]]